In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import sqlite3

BASE     = '/content/drive/MyDrive/ecommerce-etl-pipeline'
DB_PATH  = f'{BASE}/data/sql/ecommerce.db'

conn = sqlite3.connect(DB_PATH)
print("Connected to database!")

Mounted at /content/drive
Connected to database!


In [2]:
def run_query(query, title=""):
    result = pd.read_sql(query, conn)
    if title:
        print(f"\n{'='*40}")
        print(f" {title}")
        print(f"{'='*40}")
    print(result.to_string(index=False))
    return result

In [3]:
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT m.order_id)        AS total_orders,
    ROUND(SUM(m.total_revenue), 2)    AS total_revenue,
    ROUND(AVG(m.total_revenue), 2)    AS avg_order_value
FROM master_orders m
JOIN customers c
    ON m.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query, "Revenue by Customer State")


 Revenue by Customer State
customer_state  total_orders  total_revenue  avg_order_value
            SP         40494     5769081.27           142.47
            RJ         12350     2055690.45           166.45
            MG         11354     1819277.61           160.23
            RS          5344      861608.40           161.23
            PR          4923      781919.55           158.83
            SC          3546      595208.40           167.85
            BA          3256      591270.60           181.59
            DF          2080      346146.17           166.42
            GO          1957      334294.22           170.82
            ES          1995      317682.65           159.24


,customer_state,total_orders,total_revenue,avg_order_value
0,SP,40494,5769081.27,142.47
1,RJ,12350,2055690.45,166.45
2,MG,11354,1819277.61,160.23
3,RS,5344,861608.40,161.23
4,PR,4923,781919.55,158.83
5,SC,3546,595208.40,167.85
6,BA,3256,591270.60,181.59
7,DF,2080,346146.17,166.42
8,GO,1957,334294.22,170.82
9,ES,1995,317682.65,159.24


In [8]:
query= """
SELECT
    m.product_category_name_english,
    COUNT(DISTINCT m.order_id)         AS total_orders,
    ROUND(SUM(m.total_revenue), 2)    AS total_revenue,
    ROUND(AVG(r.review_score), 2)     AS avg_review_score
FROM master_orders m
JOIN reviews r
    ON m.order_id = r.order_id
GROUP BY m.product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query, "Product Category Performance")


 Product Category Performance
product_category_name_english  total_orders  total_revenue  avg_review_score
                health_beauty          8562     1406489.14              4.24
                watches_gifts          5430     1255336.62              4.13
               bed_bath_table          9072     1228526.96              4.01
               sports_leisure          7446     1118511.51              4.23
        computers_accessories          6469     1034701.97              4.08
              furniture_decor          6167      884667.37              4.07
                   housewares          5655      758784.29              4.21
                   cool_stuff          3502      687075.10              4.23
                         auto          3774      662452.31              4.15
                 garden_tools          3395      565414.90              4.19


,product_category_name_english,total_orders,total_revenue,avg_review_score
0,health_beauty,8562,1406489.14,4.24
1,watches_gifts,5430,1255336.62,4.13
2,bed_bath_table,9072,1228526.96,4.01
3,sports_leisure,7446,1118511.51,4.23
4,computers_accessories,6469,1034701.97,4.08
5,furniture_decor,6167,884667.37,4.07
6,housewares,5655,758784.29,4.21
7,cool_stuff,3502,687075.10,4.23
8,auto,3774,662452.31,4.15
9,garden_tools,3395,565414.90,4.19


In [7]:
query = """
SELECT
    seller_state,
    COUNT(DISTINCT order_id)          AS total_orders,
    ROUND(SUM(total_revenue), 2)    AS total_revenue,
    ROUND(AVG(delivery_days), 2)     AS avg_delivery_days
FROM master_orders
GROUP BY seller_state
ORDER BY total_revenue DESC
LIMIT 10;
"""

run_query(query, "Seller Performance by State")


 Seller Performance by State
seller_state  total_orders  total_revenue  avg_delivery_days
          SP         68415     9961467.89              11.90
          PR          7429     1426801.66              12.97
          MG          7647     1178846.07              12.41
          RJ          4185      911899.24              11.64
          SC          3551      716368.17              13.28
          RS          1949      429890.93              11.06
          BA           549      298571.78              13.46
          DF           803      112586.85              12.01
          PE           403      103796.23              12.40
          GO           451       77547.02              12.39


,seller_state,total_orders,total_revenue,avg_delivery_days
0,SP,68415,9961467.89,11.90
1,PR,7429,1426801.66,12.97
2,MG,7647,1178846.07,12.41
3,RJ,4185,911899.24,11.64
4,SC,3551,716368.17,13.28
5,RS,1949,429890.93,11.06
6,BA,549,298571.78,13.46
7,DF,803,112586.85,12.01
8,PE,403,103796.23,12.40
9,GO,451,77547.02,12.39


In [6]:
query = """
WITH monthly_revenue AS (
    SELECT
        order_year,
        order_month,
        order_month_year,
        COUNT(order_id)               AS total_orders,
        ROUND(SUM(total_revenue), 2)  AS revenue
    FROM master_orders
    GROUP BY order_year, order_month, order_month_year
)
SELECT
    order_month_year,
    total_orders,
    revenue,
    ROUND(revenue - LAG(revenue)
          OVER (ORDER BY order_month_year), 2) AS revenue_vs_prev_month
FROM monthly_revenue
ORDER BY order_month_year;
"""

run_query(query, "Monthly Revenue Trend")


 Monthly Revenue Trend
order_month_year  total_orders    revenue  revenue_vs_prev_month
         2016-09             1        NaN                    NaN
         2016-10           265   46566.71                    NaN
         2016-12             1      19.62              -46547.09
         2017-01           750  127545.67              127526.05
         2017-02          1653  271298.65              143752.98
         2017-03          2546  414369.39              143070.74
         2017-04          2303  390952.18              -23417.21
         2017-05          3545  566872.73              175920.55
         2017-06          3135  490225.60              -76647.13
         2017-07          3872  566403.93               76178.33
         2017-08          4193  646000.61               79596.68
         2017-09          4150  701169.99               55169.38
         2017-10          4478  751140.27               49970.28
         2017-11          7288 1153393.22              402252.95
 

,order_month_year,total_orders,revenue,revenue_vs_prev_month
0,2016-09,1,NaN,NaN
1,2016-10,265,46566.71,NaN
2,2016-12,1,19.62,-46547.09
3,2017-01,750,127545.67,127526.05
4,2017-02,1653,271298.65,143752.98
5,2017-03,2546,414369.39,143070.74
6,2017-04,2303,390952.18,-23417.21
7,2017-05,3545,566872.73,175920.55
8,2017-06,3135,490225.60,-76647.13
9,2017-07,3872,566403.93,76178.33


In [10]:
query = """
WITH repeat_customers AS (
    SELECT
        customer_id,
        COUNT(order_id) as order_count
    FROM master_orders
    GROUP BY customer_id
    HAVING COUNT(order_id) > 1
)
SELECT
    DISTINCT m.customer_state,
    rc.customer_id,
    rc.order_count
FROM repeat_customers rc
JOIN master_orders m
    ON rc.customer_id = m.customer_id
ORDER BY rc.order_count DESC;
"""

run_query(query, "Repeat Customers by State")


 Repeat Customers by State
Empty DataFrame
Columns: [customer_state, customer_id, order_count]
Index: []


,customer_state,customer_id,order_count


In [12]:
query = """
SELECT
    order_month_year,
    ROUND(SUM(total_revenue), 2)  AS monthly_revenue,
    ROUND(SUM(SUM(total_revenue))
          OVER (ORDER BY order_month_year), 2) AS running_total
FROM master_orders
GROUP BY order_month_year
ORDER BY order_month_year;
"""

run_query(query, "Running Revenue Total by Month")


 Running Revenue Total by Month
order_month_year  monthly_revenue  running_total
         2016-09              NaN            NaN
         2016-10         46566.71       46566.71
         2016-12            19.62       46586.33
         2017-01        127545.67      174132.00
         2017-02        271298.65      445430.65
         2017-03        414369.39      859800.04
         2017-04        390952.18     1250752.22
         2017-05        566872.73     1817624.95
         2017-06        490225.60     2307850.55
         2017-07        566403.93     2874254.48
         2017-08        646000.61     3520255.09
         2017-09        701169.99     4221425.08
         2017-10        751140.27     4972565.35
         2017-11       1153393.22     6125958.57
         2017-12        843199.17     6969157.74
         2018-01       1078606.86     8047764.60
         2018-02        966510.88     9014275.48
         2018-03       1120678.00    10134953.48
         2018-04       1132933.95   

,order_month_year,monthly_revenue,running_total
0,2016-09,NaN,NaN
1,2016-10,46566.71,46566.71
2,2016-12,19.62,46586.33
3,2017-01,127545.67,174132.00
4,2017-02,271298.65,445430.65
5,2017-03,414369.39,859800.04
6,2017-04,390952.18,1250752.22
7,2017-05,566872.73,1817624.95
8,2017-06,490225.60,2307850.55
9,2017-07,566403.93,2874254.48


In [13]:
query = """
SELECT
    product_category_name_english,
    ROUND(SUM(total_revenue), 2) AS category_revenue,
    RANK() OVER (ORDER BY SUM(total_revenue) DESC) AS revenue_rank
FROM master_orders
WHERE product_category_name_english != 'Unknown'
GROUP BY product_category_name_english
LIMIT 15;
"""

run_query(query, "Product Categories Ranked by Revenue")


 Product Categories Ranked by Revenue
product_category_name_english  category_revenue  revenue_rank
                health_beauty        1410846.79             1
                watches_gifts        1261634.89             2
               bed_bath_table        1224487.19             3
               sports_leisure        1119160.77             4
        computers_accessories        1030732.32             5
              furniture_decor         884202.65             6
                   housewares         760768.67             7
                   cool_stuff         693002.80             8
                         auto         669238.05             9
                 garden_tools         567606.19            10
                         toys         546445.14            11
                         baby         466249.37            12
                    perfumery         443990.70            13
                    telephony         379609.36            14
             office_furniture  

,product_category_name_english,category_revenue,revenue_rank
0,health_beauty,1410846.79,1
1,watches_gifts,1261634.89,2
2,bed_bath_table,1224487.19,3
3,sports_leisure,1119160.77,4
4,computers_accessories,1030732.32,5
5,furniture_decor,884202.65,6
6,housewares,760768.67,7
7,cool_stuff,693002.80,8
8,auto,669238.05,9
9,garden_tools,567606.19,10


In [14]:
query = """
SELECT
    customer_state,
    ROUND(SUM(total_revenue), 2) AS total_revenue,
    ROUND(SUM(total_revenue) * 100.0 /
          SUM(SUM(total_revenue)) OVER (), 2) AS revenue_pct
FROM master_orders
GROUP BY customer_state
ORDER BY total_revenue DESC;
"""

run_query(query, "Revenue Contribution by Customer State")


 Revenue Contribution by Customer State
customer_state  total_revenue  revenue_pct
            SP     5769081.27        37.41
            RJ     2055690.45        13.33
            MG     1819277.61        11.80
            RS      861608.40         5.59
            PR      781919.55         5.07
            SC      595208.40         3.86
            BA      591270.60         3.83
            DF      346146.17         2.24
            GO      334294.22         2.17
            ES      317682.65         2.06
            PE      309074.59         2.00
            CE      266463.97         1.73
            PA      212027.55         1.37
            MT      181441.72         1.18
            MA      147807.29         0.96
            PB      137834.65         0.89
            MS      134421.54         0.87
            PI      105272.17         0.68
            RN      100728.30         0.65
            AL       94195.79         0.61
            SE       70289.13         0.46
            T

,customer_state,total_revenue,revenue_pct
0,SP,5769081.27,37.41
1,RJ,2055690.45,13.33
2,MG,1819277.61,11.80
3,RS,861608.40,5.59
4,PR,781919.55,5.07
5,SC,595208.40,3.86
6,BA,591270.60,3.83
7,DF,346146.17,2.24
8,GO,334294.22,2.17
9,ES,317682.65,2.06


In [15]:
conn.close()
print("Done! All queries complete.")

Done! All queries complete.
